# 第 05 章　回归基线：最简单的模型能走多远？

> **本章问题**：先不请复杂模型出场，只报平均值或拟合一条直线，效果会怎样？

先别急着上随机森林和 XGBoost。一个复杂模型到底有没有帮助，要先有简单基准作参照。我们从训练区均值开始，再试线性回归和岭回归。后面每换一个模型，都拿同样的空间折和同一块测试区来比较。

本章关心的不只是分数，还要看预测地图和残差地图：模型把哪些地方估高了，哪些地方估低了？


## 5.1　先核对数据和固定划分

先沿用第 04 章保存的划分，这样各章的结果才可以放在一起比较。请留意两件事：

1. 这些栅格没有可靠的物理单位说明，所以指标都按“原始像元值”理解，不写 °C、mm 或 %。
2. DEM 和气温有少量缺测。填补只能在训练折里学，不能先看完整数据再切分，否则验证信息会漏进模型。

下面先确认解释器、训练测试样本数和每一折的大小。


In [ ]:
import os
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_models as gm
import geoml_style as gs

gs.setup()
T0 = time.perf_counter()

print("Python:", sys.executable)
print("Conda:", os.environ.get("CONDA_PREFIX"))
print("项目根目录:", PROJECT)

df = gd.load_splits()
print("像元总数:", len(df))
print("训练区:", int((df["split"] == "train").sum()), "| 独立测试区:", int((df["split"] == "test").sum()))

划分表里有 `split`、`fold` 和 `block` 三列，分别告诉我们样本属于训练还是测试、属于哪一折、以及来自哪个空间块。

先看表格里训练区与测试区的目标均值。测试区并不是训练区的随机缩影，这个差异会影响我们后面怎么读误差。


In [ ]:
split_table = gd.split_summary(df)
display(split_table.round(3))

train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

print("训练区 LST 均值 %.3f，测试区 LST 均值 %.3f，相差 %+.3f" % (
    train["lst"].mean(), test["lst"].mean(), test["lst"].mean() - train["lst"].mean()))
print("训练区 LST 标准差 %.3f，测试区 LST 标准差 %.3f" % (
    train["lst"].std(ddof=0), test["lst"].std(ddof=0)))

missing = pd.DataFrame({
    "训练区缺测": train[gm.FEATURES].isna().sum(),
    "测试区缺测": test[gm.FEATURES].isna().sum(),
}).rename(index=gd.FEATURE_LABELS)
display(missing)

训练区有 5796 个像元，测试区有 1360 个。空间折大小在 1010 到 1418 之间，这是按空间块平衡样本数的结果，不代表划分出错。

接下来我们让三个基线共用同一条处理流程。模型本身不同，其他设置尽量相同，这样比较才公平。


In [ ]:
models = {
    "均值基线": gm.mean_model,
    "线性回归": gm.linear_model,
    "岭回归": gm.ridge_model,
}

fold_scores = []
for name, factory in models.items():
    cv = gm.cv_scores(factory, train)
    cv.insert(0, "模型", name)
    fold_scores.append(cv)

fold_scores = pd.concat(fold_scores, ignore_index=True)
fold_scores["fold"] = fold_scores["fold"] + 1          # 展示为“空间折 1–5”
fold_scores = fold_scores.rename(columns={"fold": "空间折", "n": "验证像元数"})
fold_scores.to_csv(gd.TABLES / "05_fold_scores.csv", index=False, encoding="utf-8-sig")
display(fold_scores.round(3))

## 5.3 训练区空间 CV：模型选择只看这里

逐折分数比较碎，先按模型汇总成“折均 ± 折标准差”。这张表是**唯一允许用来比较模型的训练区证据**；测试区此时还没有打开。

In [ ]:
cv_rows = []
for name in models:
    cv = fold_scores[fold_scores["模型"] == name]
    s = gm.summarize_cv(cv, name)
    cv_rows.append(s)

cv_table = pd.DataFrame(cv_rows)
display(cv_table.round(3))

三个数字的读法：

- **MAE 折均 1.724**：均值基线平均每次差 1.72 个原始像元值。
- **RMSE 折均 2.079**：比 MAE 大，因为误差先平方再开方，大误差被放大。RMSE 与 MAE 的差距本身就是“有没有少数离谱误差”的线索。
- **R² 折均 −0.143**：负的。R² 的分母是**该验证折自身**的离差平方和，分子是模型残差平方和。均值基线用的是“另外 4 折的均值”，一旦这一折整体偏冷或偏热，它连“猜中本折均值”都不如，R² 就会小于 0。所以 R² 为负不是代码出错，而是模型没跑赢“就地取平均”。

线性回归把 MAE 从 1.724 压到 1.095（降 36.5%），RMSE 从 2.079 压到 1.339（降 35.6%），R² 从 −0.143 升到 0.304——五个环境变量确实携带了可用信息。

折间差异也值得看：线性回归在第 3 折 R² 只有 0.059，第 5 折 MAE 升到 1.391，而第 2 折 R² 有 0.431。**同一个模型在不同空间块上的表现并不一样**，只报一个总分会把这些差异藏起来。

岭回归的折均与线性回归只差在小数点后第三位（MAE 1.095、RMSE 1.339、R² 0.304）。原因很朴素：样本约 5800 个、特征只有 5 个且相关性中等，最小二乘本身就很稳，`alpha=1.0` 只能轻轻收缩系数。这里把岭回归当作“防共线性的保险”，而不是指望它提分。

## 5.4 定稿之后，测试区只打开一次

规则先说清楚：**模型形式、`alpha`、评价指标都已在训练区确定，下面这次 `fit_and_test` 是测试区唯一一次露面。** 看完测试结果不再回头改模型；否则测试区就变成了第二个验证集，报出的数字会偏乐观。

`gm.fit_and_test()` 用全部训练区像元拟合，再预测 1360 个测试像元。三个模型的预测向量按 row/col 对齐写进一张表，供复查和作图。

In [ ]:
pred_frames = {}
test_metrics = {}
metric_rows = []

for name, factory in models.items():
    _, pred, m = gm.fit_and_test(factory, train, test)
    pred_frames[name] = np.asarray(pred, dtype=float)
    test_metrics[name] = m
    metric_rows.append({"模型": name, **{f"测试_{k}": v for k, v in m.items()}})

test_pred = test[["row", "col", "x", "y", "lst"]].rename(columns={"lst": "obs"}).reset_index(drop=True)
test_pred["pred_mean"] = pred_frames["均值基线"]
test_pred["pred_linear"] = pred_frames["线性回归"]
test_pred["pred_ridge"] = pred_frames["岭回归"]
test_pred.to_csv(gd.TABLES / "05_test_predictions.csv", index=False, encoding="utf-8-sig")

test_table = pd.DataFrame(metric_rows)
metrics = cv_table.merge(test_table, on="模型")
base_mae = float(metrics.loc[metrics["模型"] == "均值基线", "测试_MAE"].iloc[0])
metrics["测试_MAE_较基线降幅"] = (base_mae - metrics["测试_MAE"]) / base_mae * 100
metrics = metrics.rename(columns={
    "MAE_折均": "CV_MAE_折均", "MAE_折标准差": "CV_MAE_折标准差",
    "RMSE_折均": "CV_RMSE_折均", "RMSE_折标准差": "CV_RMSE_折标准差",
    "R2_折均": "CV_R2_折均", "R2_折标准差": "CV_R2_折标准差",
    "测试_MAE_较基线降幅": "测试_MAE_较基线降幅/%",
})
metrics = metrics[[
    "模型", "CV_MAE_折均", "CV_MAE_折标准差", "CV_RMSE_折均", "CV_RMSE_折标准差",
    "CV_R2_折均", "CV_R2_折标准差", "测试_MAE", "测试_RMSE", "测试_R2", "测试_MAE_较基线降幅/%",
]]
metrics.to_csv(gd.TABLES / "05_metrics.csv", index=False, encoding="utf-8-sig")
display(metrics.round(3))

测试区的结果：

| 模型 | 测试 MAE | 测试 RMSE | 测试 R² |
| --- | --- | --- | --- |
| 均值基线 | 1.733 | 2.079 | −0.218 |
| 线性回归 | 1.351 | 1.620 | 0.260 |
| 岭回归 | 1.351 | 1.621 | 0.260 |

线性（岭）回归相对均值基线：MAE 降低 0.382（22.0%），RMSE 降低 0.458（22.0%）。但和自己在训练区 CV 上的 1.095 / 1.339 / 0.304 比，测试区每一项都更差：MAE 高 0.256，R² 低 0.044。这不是代码矛盾，而是**测试区与训练区分布不同**：测试区 LST 均值 12.83，训练区 11.95，整体偏暖约 0.88；测试区的 DEM 均值更低（175.7 对 184.2）、气温更高（4.18 对 3.85）、降水更少（648.5 对 667.9）。空间 CV 估的是“训练区内部换一块地”的表现，换到另一块气候与地形条件不同的区域，误差自然上升。这也是本案例只能定位为**留出区域估计的方法演示**、不能当作甘南县 LST 产品的原因。

均值基线的测试 R² 是 **−0.218**，可以这样验算：它把所有像元都预测成训练均值 11.948，测试区标准差是 1.883，`R² = 1 − (RMSE/σ)² = 1 − (2.079/1.883)² ≈ −0.218`。用测试集自己的均值去预测，R² 才会恰好为 0；基线用的是训练均值，所以为负。线性回归的 R² 也可以同样验算：`1 − (1.621/1.883)² ≈ 0.260`。

三个指标各司其职：MAE 回答“典型误差多大”，RMSE 回答“大误差有多严重”，R² 回答“比直接猜测试区均值好多少”。R² 的分母依赖测试集自身的离散程度，**只适合同一测试集内比较**，不能跨数据集比大小。

## 5.5 同一窗口的三联图：误差长什么样

指标只说大小，不说位置。把测试区裁成 36×41 的窗口，用 `gd.test_window()` 取出观测、用 `gd.grid_from_predictions()` 把预测向量放回网格，就能画“观测—预测—残差”。

三张图遵守同一套规则：都用这个留出范围；观测与预测共用色标（测试区观测的最小值到最大值）；残差图以 0 为中心，色标标注“预测 − 观测”；窗口内没有有效像元的位置是浅灰；坐标是经纬度，并加比例尺与指北针。

In [ ]:
window = gd.test_window(df)
print("窗口行范围 %d–%d，列范围 %d–%d，网格 %d×%d，有效像元 %d" % (
    window["row0"], window["row1"], window["col0"], window["col1"],
    window["shape"][0], window["shape"][1], int(np.isfinite(window["obs"]).sum())))

# 这个窗口只跨 0.37° 经度，共享样式的一位小数刻度会重复显示（123.4°E 出现两次）。
# 仅在本次运行中把经纬度刻度改为两位小数，不改动 src/geoml_style.py。
import matplotlib.ticker as mticker

def degree_ticks(ax, crs):
    if crs is None:
        return
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.1))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f°E"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f°N"))

gs._format_coord_axes = degree_ticks

triptych_files = [
    ("均值基线", "05_triptych_mean.png"),
    ("线性回归", "05_triptych_linear.png"),
    ("岭回归", "05_triptych_ridge.png"),
]
for name, fname in triptych_files:
    pred_grid = gd.grid_from_predictions(window, pred_frames[name])
    resid = pred_grid - window["obs"]
    path = gs.plot_triptych(
        window["obs"], pred_grid, resid, window["extent"],
        title=f"{name}：独立测试区（中南连续区块，{len(test)} 个像元）",
        out_name=fname,
        cbar_label="LST（原始像元值）",
        cmap="magma",
        crs="EPSG:4326",
        add_bar=True,
    )
    print("已导出:", path.name)

先看观测面板：测试区 LST 大致在 6.2–17.9 之间，南部与中部偏暖，夹杂着几处冷斑。

再看预测面板，三张图的色标与观测完全相同，所以可以直接比颜色：

- **均值基线**是一整块均匀的 11.95，没有任何空间纹理——这正是“残差图”里出现大面积同色块的原因：预测 − 观测在全区平均为 −0.879，等于把测试区整体报冷了。残差图不是随机噪点，而是一张“地形/温度格局的反色照片”。
- **线性回归与岭回归**的预测范围只有 10.31–13.97，远窄于观测的 6.24–17.88。模型抓住了大尺度的冷暖梯度（北部偏冷、南部偏暖），但把极端值都拉向中间：最冷的像元被高估，最暖的像元被低估。这就是线性模型的“回归到均值”现象，散点图里会看得更清楚。
- 残差图（预测 − 观测）的主体在北部偏蓝、南部偏红，说明误差带有明显的**空间组织**；但西北角强冷斑附近是一大片红色，因为模型把这块冷斑“填平”了——预测远高于观测。两类偏差方向相反，却都不是随机噪点。

## 5.6 模型对比：同一把尺子

把 5 个空间折的 RMSE 画成点、折均画成横线，测试区画成菱形，三个模型共用一条纵轴。这样既能看到模型间的差距，也能看到“测试区比 CV 难多少”。

In [ ]:
fold_rmse = {name: fold_scores.loc[fold_scores["模型"] == name, "RMSE"].to_numpy() for name in models}
test_rmse = {name: test_metrics[name]["RMSE"] for name in models}

compare_path = gs.model_comparison(
    fold_rmse, test_rmse, "05_model_compare.png",
    "三个基线模型的 RMSE：5 个空间折与独立测试区",
    ylabel="RMSE（原始像元值）",
)
print("已导出:", compare_path.name)

三个模型的测试菱形都落在各自折均横线的**上方**：均值基线折均 1.724、测试 2.079；线性回归折均 1.339、测试 1.621；岭回归与线性几乎重合。这再次说明测试区整体更难，不能把 CV 分数当成外推精度的无偏估计。

模型之间的差距则非常稳定：线性/岭回归的 5 个折点全部低于均值基线的折点，测试区也一样。这种“每一折都赢”比“平均分略高”更有说服力。

## 5.7 残差不是随机噪点

如果残差只是白噪声，相邻像元的残差应该互不相关。用一个简单的 rook 邻接（上下左右四个邻居）计算残差的 Moran's I，再按南北行带统计平均残差。

In [ ]:
def moran_rook(grid):
    z = np.asarray(grid, dtype=float)
    valid = np.isfinite(z)
    zc = z - z[valid].mean()
    num, pairs = 0.0, 0
    for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        a = zc[max(0, dr):z.shape[0] + min(0, dr), max(0, dc):z.shape[1] + min(0, dc)]
        b = zc[max(0, -dr):z.shape[0] + min(0, -dr), max(0, -dc):z.shape[1] + min(0, -dc)]
        ok = np.isfinite(a) & np.isfinite(b)
        num += float(np.sum(a[ok] * b[ok]))
        pairs += int(ok.sum())
    return num / (pairs * z[valid].var())

ridge_grid = gd.grid_from_predictions(window, pred_frames["岭回归"])
ridge_resid = ridge_grid - window["obs"]
print("岭回归测试区残差：均值 %+.3f，标准差 %.3f，Moran's I（rook）= %.3f" % (
    np.nanmean(ridge_resid), np.nanstd(ridge_resid, ddof=1), moran_rook(ridge_resid)))

resid_df = test_pred.assign(残差=test_pred["pred_ridge"] - test_pred["obs"])
resid_df["行带"] = pd.cut(resid_df["row"], bins=[54, 63, 72, 81, 91],
                          labels=["55–63（北）", "64–72", "73–81", "82–90（南）"])
band = resid_df.groupby("行带", observed=True).agg(
    像元数=("残差", "size"),
    平均残差=("残差", "mean"),
    平均绝对误差=("残差", lambda s: s.abs().mean()),
)
display(band.round(3))

slope = np.polyfit(test_pred["obs"], test_pred["pred_ridge"], 1)[0]
print("预测对观测的回归斜率: %.3f" % slope)
print("残差与观测的相关系数: %.3f" % np.corrcoef(resid_df["残差"], test_pred["obs"])[0, 1])
print("预测范围 %.2f–%.2f，观测范围 %.2f–%.2f" % (
    test_pred["pred_ridge"].min(), test_pred["pred_ridge"].max(),
    test_pred["obs"].min(), test_pred["obs"].max()))

Moran's I = 0.862（接近 1 表示强正空间自相关），残差在空间上是成片出现的。按“预测 − 观测”的定义，南北行带的平均残差从北段的 −0.71 变到南段的 +0.70：模型在北部报得比实际**低**（偏冷），到了南部又报得比实际**高**（偏暖）。同一套线性系数无法同时适配窗口两端的温度水平，误差沿南北方向系统性地翻转，而不是随机散布。

预测对观测的回归斜率只有 0.183（理想应为 1），残差与观测的相关系数为 −0.949：观测越暖，模型越是低估；观测越冷，模型越高估。这是典型的“向均值收缩”，也是线性模型在 6.2–17.9 这样大的温度跨度上只能取到 10.3–14.0 的原因。

需要提醒：残差高度自相关意味着 1360 个像元携带的独立信息远少于 1360 份。本章没有计算置信区间；如果后续要报不确定性，应按空间块（而不是像元）做重采样，否则区间会过窄。

## 5.8 地理视角

- 测试区位于县域中南部，DEM 更低、气温更高、降水更少，LST 均值比训练区高 0.88。**空间划分不是随机抽样**，它模拟的是“把模型搬到另一块地方用”，因此比随机 CV 更接近真实外推场景，也更严格。
- 线性模型的主要信息来源是气温（标准化系数 +0.98）和高程（+0.31）；NDVI 为 +0.15、降水为 −0.26，土壤水分接近 0。这不是说植被不重要，而是说在这 5 个变量、这个尺度、这条线性假设下，植被的独立贡献很弱；变量之间还彼此相关，系数不能当因果效应读。
- 残差图里成片的冷暖偏差，多半来自地形与下垫面的非线性组合（例如阳坡/阴坡、河谷、灌溉地块），这些是线性项表达不了的。第 6 章的树模型正是为此而来。

## 5.9 常见坑

1. **先看测试区再回头调参。** 只要根据测试分数改过模型，测试区就不再独立，报出的 R² 会偏乐观。正确顺序是：训练区 CV 定稿 → 测试区评估一次 → 不再改动。
2. **在划分之前填补或标准化。** `SimpleImputer`、`StandardScaler` 必须放进 `Pipeline`，让 `fit` 只见到训练折；否则验证折的均值/中位数会泄漏进训练。
3. **看到负 R² 就以为代码错了。** R² 的分母是评估集自身的方差；用训练均值去预测一个整体偏暖的测试区，R² 本来就会为负。
4. **把原始像元值当 °C。** 元数据没有给出可靠单位，本章所有数字都是原始像元值，不能换算成温度。
5. **拿随机 CV 的分数估计空间外推能力。** 第 4 章已经演示过随机划分会因空间自相关而偏乐观；本章只信空间折。
6. **残差图色标不居中对 0。** 若用普通连续色标，正负偏差会共用一端颜色，看不出“整体偏冷还是偏暖”。本章统一用 `TwoSlopeNorm` 以 0 为中心。

## 5.10 练习

**只用气温（`tem`）一个特征**，重跑线性回归的 5 折空间 CV 与独立测试区评估，把 MAE、RMSE、R² 与本章五变量线性回归对比。提示：`gm.cv_scores()` 和 `gm.fit_and_test()` 都有 `features` 参数，传入 `["tem"]` 即可；测试区仍然只评估一次。完成后回答：少用四个变量，误差增加了多少？这是否说明那四个变量没有信息？

## 小结

- 均值基线在测试区的 MAE 1.733、R² −0.218；它连“就地取平均”都不如，因为测试区比训练区暖 0.88。基线的作用是给出“零信息”的下界。
- 五变量线性回归把测试 MAE 降到 1.351（−22.0%）、R² 提到 0.260。这是本案例的**线性上限参考**：五个变量确实有信息，但远远不够解释全部温度差异。
- 岭回归（`alpha=1.0`）与线性回归几乎完全一致（测试 MAE 差不到 0.001）。在样本充足、特征少且相关性中等时，正则化不是提分手段。
- 测试区比空间 CV 更难（MAE 1.095 → 1.351），差异来自训练区与测试区的分布不同；这正是“留出区域估计”要如实报告的部分。
- 残差 Moran's I 0.862、预测—观测斜率 0.183，说明线性模型漏掉了成片的空间结构，为第 6 章的树模型留下了明确的改进空间。

In [ ]:
scatter_path = gs.plot_scatter(
    test_pred["obs"], test_pred["pred_ridge"], "05_scatter_ridge.png",
    "岭回归：独立测试区观测—预测",
    r2=test_metrics["岭回归"]["R2"], mae=test_metrics["岭回归"]["MAE"],
)
print("已导出:", scatter_path.name)

散点没有整体偏离 1:1 线（平均残差 +0.013），但形状明显被压扁：低观测值被抬到 10 以上，高观测值被压在 14 以下。误差主要来自“幅度不够”，而不是“方向错了”。

In [ ]:
elapsed = time.perf_counter() - T0

def r4(x):
    return round(float(x), 4)

summary_json = {
    "notebook": "05_回归基线.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "run_seconds": round(elapsed, 1),
    "n_train": int(len(train)),
    "n_test": int(len(test)),
    "test_block": {"row": [int(window["row0"]), int(window["row1"])],
                   "col": [int(window["col0"]), int(window["col1"])]},
    "mean_lst_train": r4(train["lst"].mean()),
    "mean_lst_test": r4(test["lst"].mean()),
    "metrics": {
        name: {
            "cv_mae_mean": r4(cv_table.loc[cv_table["模型"] == name, "MAE_折均"].iloc[0]),
            "cv_mae_sd": r4(cv_table.loc[cv_table["模型"] == name, "MAE_折标准差"].iloc[0]),
            "cv_rmse_mean": r4(cv_table.loc[cv_table["模型"] == name, "RMSE_折均"].iloc[0]),
            "cv_rmse_sd": r4(cv_table.loc[cv_table["模型"] == name, "RMSE_折标准差"].iloc[0]),
            "cv_r2_mean": r4(cv_table.loc[cv_table["模型"] == name, "R2_折均"].iloc[0]),
            "cv_r2_sd": r4(cv_table.loc[cv_table["模型"] == name, "R2_折标准差"].iloc[0]),
            "test_mae": r4(test_metrics[name]["MAE"]),
            "test_rmse": r4(test_metrics[name]["RMSE"]),
            "test_r2": r4(test_metrics[name]["R2"]),
        }
        for name in models
    },
    "ridge_alpha": 1.0,
    "residual_moran_rook": r4(moran_rook(ridge_resid)),
    "pred_obs_slope_ridge": r4(slope),
    "figures": [
        "05_triptych_mean.png", "05_triptych_linear.png", "05_triptych_ridge.png",
        "05_model_compare.png", "05_scatter_ridge.png",
    ],
    "tables": [
        "05_metrics.csv", "05_fold_scores.csv", "05_test_predictions.csv", "05_run_summary.json",
    ],
    "unverified": [
        "LST、气温、降水、土壤水分的单位未在元数据中注明，全部按原始像元值处理",
        "LST 为 2021 年（月日未知），降水与土壤水分为 2020 年，时间不完全同期",
        "案例定位为留出区域估计的方法演示，不是甘南县 LST 产品，不作因果解释",
    ],
}
gd.write_json(summary_json, "05_run_summary.json")
print("运行耗时 %.1f s" % elapsed)
print("已写出 outputs/tables/05_run_summary.json")